# 2 Training (Stufe 1, V1)

Ein U-Net sagt aus einem 256-px-Ausschnitt eine **Heatmap** der Grenzpunktsymbole voraus: pro Label eine Gaussglocke mit σ = 2 px.
Aufbau nach den PyTorch-Tutorials (*Datasets & DataLoaders*, *Quickstart*) und dem Quickstart von `segmentation_models_pytorch`.

Das Dataset (`PunktDataset`) steht in `grenzpunkte.py`: Die DataLoader-Worker laufen unter macOS als eigene Prozesse und finden eine Klasse nur in einer `.py`-Datei, nicht in einer Notebook-Zelle.

Ergebnis: `unet.pt` (Modell mit dem kleinsten Val-Loss) und `train_verlauf.csv` (Loss pro Epoche).
Dauer: etwa 3 Stunden für 20 Epochen auf dem M3 Pro.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from torch import nn
from torch.utils.data import DataLoader

from grenzpunkte import BATCH_SIZE, PunktDataset, device

EPOCHS = 20
LERNRATE = 1e-4

## Daten
Augmentiert wird nur im Training: Helligkeit, Kontrast und Farbe, dazu 90°-Drehungen und Spiegelungen. Die Heatmap wird mitgedreht und mitgespiegelt, statt die Punkte umzurechnen.

In [ ]:
train_dataloader = DataLoader(PunktDataset("train", augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_dataloader = DataLoader(PunktDataset("val"), batch_size=BATCH_SIZE, num_workers=4)
print(f"Training: {len(train_dataloader.dataset)} Ausschnitte, Validierung: {len(val_dataloader.dataset)}")

## Ein Beispiel ansehen
Eingabe (für die Anzeige wieder entnormiert) und Ziel-Heatmap eines augmentierten Trainingsausschnitts.

In [ ]:
X, y = train_dataloader.dataset[0]
bild = X.permute(1, 2, 0) * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(bild.clamp(0, 1))
axes[1].imshow(y[0], cmap="Blues")
for ax, titel in zip(axes, ["Eingabe", "Ziel-Heatmap"]):
    ax.set_title(titel)
    ax.axis("off")

## Modell, Loss und Optimizer
U-Net mit ResNet34-Encoder (ImageNet-Gewichte). Der Hintergrund überwiegt stark, darum werden die positiven Pixel im BCE-Loss 100-fach gewichtet (`pos_weight`).

In [ ]:
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(100.0, device=device))
optimizer = torch.optim.Adam(model.parameters(), lr=LERNRATE)

## Trainings- und Testschleife
Wie im PyTorch-Quickstart. Beide Funktionen geben den mittleren Loss der Epoche zurück.

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    summe = 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        summe += loss.item()
        if batch % 100 == 0:
            print(f"loss: {loss.item():>7f}  [{(batch + 1) * len(X):>5d}/{size:>5d}]")
    return summe / len(dataloader)


def test(dataloader, model, loss_fn):
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            test_loss += loss_fn(model(X), y).item()
    test_loss /= len(dataloader)
    print(f"Val loss: {test_loss:>8f}\n")
    return test_loss

## Training
Nach jeder Epoche wird der Val-Loss berechnet. Das Modell mit dem kleinsten Val-Loss wird als `unet.pt` gespeichert, der Verlauf als `train_verlauf.csv` (für die Grafiken in `06_grafiken.ipynb`).

In [ ]:
verlauf = []
best = float("inf")
for t in range(EPOCHS):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss = train(train_dataloader, model, loss_fn, optimizer)
    val_loss = test(val_dataloader, model, loss_fn)
    verlauf.append({"epoche": t + 1, "train_loss": train_loss, "val_loss": val_loss})
    pd.DataFrame(verlauf).to_csv("train_verlauf.csv", index=False)
    if val_loss < best:
        best = val_loss
        torch.save(model.state_dict(), "unet.pt")
print("Done!")